In [1]:
import requests
from bs4 import BeautifulSoup
from selenium.webdriver import Chrome
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as expected_conditions

In [2]:
import re
import json
from pathlib import Path

# --- Regexes ---
# Company name: first non-empty line after optional "View Summary"
COMPANY_RE = re.compile(r'^\s*(?:View Summary\s*)?\n\s*([^\n]+?)\s*$', re.MULTILINE)

# Speaker line examples:
# "Scott BibaudExecutive"
# "Mike Bishop;BishopIRAttendee"
# "OperatorOperator"
#
# This pattern grabs:
#  - speaker "name" part (can include spaces, dots, apostrophes, hyphens, semicolons)
#  - optional role suffix: Operator | Executive | Attendee
#
# It expects the speaker line to be a whole line by itself.
SPEAKER_LINE_RE = re.compile(
    r'^(?P<speaker>[A-Za-z][A-Za-z .,\-\'&;/]+?)\s*(?P<role>Operator|Executive|Attendee)?\s*$',
    re.MULTILINE
)

def extract_company(text: str) -> str:
    # Prefer the line immediately after "View Summary" if present
    m = re.search(r'View Summary\s*\n\s*([^\n]+)\s*', text, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # Fallback: first non-empty line
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines[0] if lines else ""

def split_by_speaker(text: str):
    matches = list(SPEAKER_LINE_RE.finditer(text))
    segments = []

    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        speaker = m.group("speaker").strip()
        role = (m.group("role") or "").strip()

        speech = text[start:end].strip()
        if not speech:
            continue

        segments.append({
            "speaker": speaker,
            "role": role,
            "text": speech
        })

    return segments

def process_folder(input_dir: str, output_dir: str):
    in_path = Path(input_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    for txt_file in in_path.glob("*.txt"):
        text = txt_file.read_text(encoding="utf-8", errors="ignore")

        company = extract_company(text)
        segments = split_by_speaker(text)

        result = {
            "company": company,
            "file": txt_file.name,
            "segments": segments
        }

        out_file = out_path / (txt_file.stem + ".json")
        out_file.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Done. Wrote JSON files to: {out_path.resolve()}")

if __name__ == "__main__":
    # Change these paths
    process_folder(
        input_dir="/Users/eduardo/Desktop/transcripts",
        output_dir="/Users/eduardo/Desktop/output"
    )

Done. Wrote JSON files to: /Users/eduardo/Desktop/output


In [3]:
import re
import json
from pathlib import Path

# -----------------------------
# Token counting (tiktoken if available, else approximation)
# -----------------------------
def token_len(s: str) -> int:
    try:
        import tiktoken
        enc = tiktoken.get_encoding("cl100k_base")
        return len(enc.encode(s))
    except Exception:
        # fallback: ~4 chars/token (rough but workable)
        return max(1, len(s) // 4)

# -----------------------------
# Sentence splitting (simple, natural endpoints)
# -----------------------------
SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9"“‘\(\[])')

def split_into_sentences(text: str):
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    return SENT_SPLIT_RE.split(text)

# -----------------------------
# Chunk sentences under token limit
# -----------------------------
def chunk_sentences(sentences, max_tokens: int):
    chunks = []
    cur = []
    cur_tokens = 0

    for sent in sentences:
        t = token_len(sent)
        # If a single sentence is too big, hard-split by words
        if t > max_tokens:
            words = sent.split()
            buf = []
            for w in words:
                candidate = (" ".join(buf + [w])).strip()
                if token_len(candidate) > max_tokens and buf:
                    chunks.append(" ".join(buf))
                    buf = [w]
                else:
                    buf.append(w)
            if buf:
                chunks.append(" ".join(buf))
            continue

        candidate_text = (" ".join(cur + [sent])).strip()
        if token_len(candidate_text) <= max_tokens:
            cur.append(sent)
            cur_tokens = token_len(candidate_text)
        else:
            if cur:
                chunks.append(" ".join(cur).strip())
            cur = [sent]
            cur_tokens = t

    if cur:
        chunks.append(" ".join(cur).strip())

    return chunks

# -----------------------------
# Company + speaker parsing (same style as before)
# -----------------------------
SPEAKER_LINE_RE = re.compile(
    r'^(?P<speaker>[A-Za-z][A-Za-z .,\-\'&;/]+?)\s*(?P<role>Operator|Executive|Attendee)?\s*$',
    re.MULTILINE
)

def extract_company(text: str) -> str:
    m = re.search(r'View Summary\s*\n\s*([^\n]+)\s*', text, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines[0] if lines else ""

def split_by_speaker(text: str):
    matches = list(SPEAKER_LINE_RE.finditer(text))
    segments = []

    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        speaker = m.group("speaker").strip()
        role = (m.group("role") or "").strip()
        speech = text[start:end].strip()

        if not speech:
            continue

        segments.append({"speaker": speaker, "role": role, "text": speech})

    return segments

# -----------------------------
# Classification: Prepared / Q / A / O
# -----------------------------
QA_TRIGGER_RE = re.compile(r'\b(q&a|question[s]?)\b', re.IGNORECASE)
OPERATOR_Q_TRIGGER_RE = re.compile(r'\b(our first question|next question|question will come from)\b', re.IGNORECASE)

def classify_segments(segments):
    """
    Rules (simple + robust):
      - Operator => O
      - Before Q&A starts => Prepared (for non-Operator)
      - Q&A starts when Operator mentions questions or "Q&A"
      - After Q&A starts:
          * Attendee => Q
          * Executive => A
          * otherwise => Prepared (fallback)
    """
    qa_started = False
    classified = []

    for seg in segments:
        speaker = seg["speaker"]
        role = seg["role"]
        text = seg["text"]

        is_operator = (role == "Operator") or (speaker.strip().lower() == "operator")

        if is_operator:
            cls = "O"
            # Operator can signal Q&A start
            if QA_TRIGGER_RE.search(text) or OPERATOR_Q_TRIGGER_RE.search(text):
                qa_started = True
        else:
            if not qa_started:
                cls = "Prepared"
                # Non-operator can also mention "we will now take questions"
                if QA_TRIGGER_RE.search(text) and re.search(r'\btake question', text, re.IGNORECASE):
                    qa_started = True
            else:
                if role == "Attendee":
                    cls = "Q"
                elif role == "Executive":
                    cls = "A"
                else:
                    cls = "Prepared"

        classified.append({**seg, "class": cls})

    return classified

# -----------------------------
# Split long sequences respecting token limits
# -----------------------------
def split_sections(classified_segments, max_tokens: int):
    """
    Returns a flat list of sections after splitting.
    Each output section corresponds to one original segment, possibly split into multiple parts.
    """
    out = []
    section_id = 0

    for seg in classified_segments:
        sentences = split_into_sentences(seg["text"])
        chunks = chunk_sentences(sentences, max_tokens=max_tokens)

        for part_idx, chunk in enumerate(chunks, start=1):
            section_id += 1
            out.append({
                "section_id": section_id,
                "speaker": seg["speaker"],
                "role": seg["role"],
                "class": seg["class"],          # Prepared / Q / A / O
                "part": part_idx,
                "parts_total": len(chunks),
                "token_len": token_len(chunk),
                "text": chunk
            })

    return out

# -----------------------------
# Folder processing
# -----------------------------
def process_folder(input_dir: str, output_dir: str, max_tokens: int = 1500):
    in_path = Path(input_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    for txt_file in in_path.glob("*.txt"):
        text = txt_file.read_text(encoding="utf-8", errors="ignore")

        company = extract_company(text)
        segments = split_by_speaker(text)
        classified = classify_segments(segments)
        sections = split_sections(classified, max_tokens=max_tokens)

        result = {
            "company": company,
            "file": txt_file.name,
            "max_tokens": max_tokens,
            "num_original_segments": len(segments),
            "num_sections_after_split": len(sections),
            "sections": sections
        }

        out_file = out_path / (txt_file.stem + ".json")
        out_file.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Done. Wrote JSON files to: {out_path.resolve()}")

if __name__ == "__main__":
    process_folder(
        input_dir="path/to/your/txt_folder",
        output_dir="path/to/output_json",
        max_tokens=1500
    )

Done. Wrote JSON files to: /Users/eduardo/Desktop/Python_Coding/Applied-Data-Science-Classes/BigData/path/to/output_json


In [4]:
pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 1.1 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 1.2 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 1.2 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 1.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sentence_transformers]ence_transformers]
Note: you may need to restart the kernel to use updated packages.
